In [27]:
from datasets import load_from_disk

In [29]:
ds = load_from_disk("dataset/parcanDeb-rec-split")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 7713
    })
    dev: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 987
    })
    test: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 983
    })
    tiny: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 50
    })
})


In [38]:
print(ds["train"][0]["Text"])

Esta sesión del parlamento se realizó el 2007-01-29. · 6L/PPL-0018 Debate de toma en consideración. Proposición de Ley, de los Grupos Parlamen­tarios Coalición Canaria (CC), Socialista Canario y Mixto, para la Modificación Parcial de la Ley 1/1998, de 8 de enero, de Régimen Jurídico de los Espectáculos Públicos y Actividades Clasificadas, sobre Régimen Especial para las Actividades y Espectáculos que se Desarrollen en Determinados Festejos Populares. El señor presidente: Un único punto del orden del día, debate de toma en consideración de la proposición de Ley... (Pausa.) Debate de toma en consideración de la proposición de Ley de los Grupos Parlamentarios Coalición Canaria, Socialista Canario y Mixto, para la Modificación Parcial de la Ley 1/1998, de 8 de enero, de Régimen Jurídico de los Espectáculos Públicos y Actividades Clasificadas, sobre Régimen Especial para las Actividades y Espectáculos que se Desarrollen en Determinados Festejos Populares. Ruego al señor secretario segundo d

In [32]:
print(ds["train"][0]["Speakers"])

['Benítez De Lugo Massieu', 'Hernández Spínola', 'Rodríguez Pérez', 'Belda Quintana']


In [37]:
inter = ds["train"][0]["Interventions"]
for i, intervention in enumerate(inter):
    print(f"Intervention {i}:")
    print(intervention)
    print("-----")


Intervention 0:
['"El Gobierno, tras deliberar, y a propuesta del consejero de Presidencia y Justicia, acuerda manifestar el criterio favorable respecto a la toma en consideración y la conformidad a la tramitación de la proposición de Ley para la Modificación Parcial de la Ley 1/1998, de 8 de enero, de Régimen Jurídico de los Espectáculos Públicos y Actividades Clasificadas, sobre Régimen Especial para las Actividades y Espectáculos que se Desarrollen en Determinados Festejos Populares, presentada por los Grupos Parlamentarios de Coalición Canaria, Socialista Canario y Mixto." El señor presidente: Muchas gracias, señor secretario segundo. Para un turno de fijación de posición por los distintos grupos parlamentarios, ¿el Grupo Mixto? (Pausa.) No va a intervenir. ¿El Grupo Parlamentario Socialista? Tiene la palabra el señor Hernández Spínola.']
-----
Intervention 1:
['Buenas tardes. Señor presidente, señorías. El Grupo Parlamentario Socialista apoya esta iniciativa parlamentaria, suscrit

In [35]:
vector = ds["train"][0]["label"]
# cuenta la cantidad de elementos no nulos
non_zero_count = sum(1 for v in vector if v != 0)
print(f"Cantidad de elementos no nulos en el vector de etiquetas: {non_zero_count}")

Cantidad de elementos no nulos en el vector de etiquetas: 4


In [42]:
import numpy as np
# --- CÁLCULO AUTOMÁTICO DEL PRIOR (Insertar antes del Trainer) ---
print("\n[-] Calculando el Prior REAL de la estrategia actual...")
total_ones = 0
total_elements = 0
train_dataset = ds['train']

# Iteramos sobre una muestra del dataset procesado (o todo si es rápido)
# train_dataset ya tiene los vectores "inflados" con all_participants
for i in range(min(1000, len(train_dataset))): # Muestreamos 1000 ejemplos para ir rápido
    labels = np.array(train_dataset[i]['label']) # Esto es un tensor
    total_ones += labels.sum().item()
    total_elements += len(labels)

real_prior = total_ones / total_elements
print(f"    Prior Observado: {real_prior:.5f} (es decir, {real_prior*100:.2f}%)")
print(f"    Recomendación para nnPU: Usar pi = {real_prior * 1.1:.5f} (un poco de margen)")

# Sobreescribimos el valor para usarlo en el Trainer
prior_to_use = real_prior * 1.1


[-] Calculando el Prior REAL de la estrategia actual...
    Prior Observado: 0.01106 (es decir, 1.11%)
    Recomendación para nnPU: Usar pi = 0.01216 (un poco de margen)
